In [1]:
import requests

In [2]:
url = 'https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp/events'

response = requests.get(url=url)
response.json()


[{'id': '9272499487',
  'type': 'WatchEvent',
  'actor': {'id': 166287596,
   'login': 'iamironman07',
   'display_login': 'iamironman07',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/iamironman07',
   'avatar_url': 'https://avatars.githubusercontent.com/u/166287596?'},
  'repo': {'id': 419661684,
   'name': 'DataTalksClub/data-engineering-zoomcamp',
   'url': 'https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp'},
  'payload': {'action': 'started'},
  'public': True,
  'created_at': '2026-05-09T18:37:10Z',
  'org': {'id': 72699292,
   'login': 'DataTalksClub',
   'gravatar_id': '',
   'url': 'https://api.github.com/orgs/DataTalksClub',
   'avatar_url': 'https://avatars.githubusercontent.com/u/72699292?'}},
 {'id': '9270956232',
  'type': 'WatchEvent',
  'actor': {'id': 53072029,
   'login': 'kapoolay',
   'display_login': 'kapoolay',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/kapoolay',
   'avatar_url': 'https://avatars.githubuserco

To check how many requests we can make with this git API

In [3]:
url = 'https://api.github.com/rate_limit'

requests.get(url=url).json()

{'resources': {'code_search': {'limit': 60,
   'remaining': 59,
   'reset': 1778361674,
   'used': 1,
   'resource': 'code_search'},
  'core': {'limit': 60,
   'remaining': 59,
   'reset': 1778361674,
   'used': 1,
   'resource': 'core'},
  'graphql': {'limit': 0,
   'remaining': 0,
   'reset': 1778361973,
   'used': 0,
   'resource': 'graphql'},
  'integration_manifest': {'limit': 5000,
   'remaining': 5000,
   'reset': 1778361973,
   'used': 0,
   'resource': 'integration_manifest'},
  'search': {'limit': 10,
   'remaining': 10,
   'reset': 1778358433,
   'used': 0,
   'resource': 'search'}},
 'rate': {'limit': 60,
  'remaining': 59,
  'reset': 1778361674,
  'used': 1,
  'resource': 'core'}}

In [ ]:
import time
# Let's Check how much limit we have
remaining = requests.get(url=url).json()['rate']['remaining']

# So the access becomes zero then we have to wait 60 mins, so we can give just wait time
if remaining == 0:
    time.sleep(30)

In [ ]:
# This required the API token
url = "https://api.github.com/user"
requests.get(url=url).json()

{'message': 'Requires authentication',
 'documentation_url': 'https://docs.github.com/rest',
 'status': '401'}

In [ ]:
API_TOKEN = "YOUR_ACCESS_TOKEN"
headers = {
    'Authorization': f'Bearer {API_TOKEN}'
}

requests.get(url, headers=headers).json()

Handle the Pagination method, means if there are many pages then can handle with specific keywords

so we can break the while loop, in our case there is next button so we can check when the next button is not visible 
just break the loop

In [6]:
required_url = 'https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp/events'

response = requests.get(url=required_url)
response

<Response [200]>

In [7]:
response.headers['link']

'<https://api.github.com/repositories/419661684/events?page=2>; rel="next", <https://api.github.com/repositories/419661684/events?page=10>; rel="last"'

In [8]:
# Let's check all the pages
response.links['next']['url']

'https://api.github.com/repositories/419661684/events?page=2'

In [9]:
required_url = 'https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp/events'


while True:
    response = requests.get(url=required_url)
    data = response.json()
    print(len(data))
    
    if 'next' not in response.links:
        break
    
    url = response.links['next']['url'] 

30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
2


In [10]:
# Now let's not get all the page data, we will devide it into the chunks, so it will not store large data

def events_getter():
    required_url = 'https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp/events'


    while True:
        response = requests.get(url=required_url)
        data = response.json()
        yield data # It will return everytime when we reuest a new page, yield works as a return
        
        if 'next' not in response.links:
            break
        
        url = response.links['next']['url'] 

In [11]:
events_pages = events_getter()

for element in events_pages:
    print(element)

{'message': "API rate limit exceeded for 193.148.48.69. (But here's the good news: Authenticated requests get a higher rate limit. Check out the documentation for more details.)", 'documentation_url': 'https://docs.github.com/rest/overview/resources-in-the-rest-api#rate-limiting'}


Normalization Example

In [ ]:
event = events_pages[0]
event

In [ ]:
def process_event(event):
    result = {}
    
    result['id'] = event['id']
    result['type'] = event['type']
    result['public'] = event['public']
    result['create_at'] = event['create_at']
    
    result['actor____id'] = event['actor']['id']
    result['actor____login'] = event['actor']['login']
    
    return result

In [ ]:
# below is for the one page
# processed_event = process_event(event=event)

# for all the pages
# we will stor the values in list format
processed_event = []
for event in events_pages:
    processed_event = process_event(event=event)
    process_event.append(processed_event)

Adjust the time format in this API data

In [ ]:
from datetime import datetime
# datetime.fromisoformat()

def process_event(event):
    result = {}
    
    result['id'] = event['id']
    result['type'] = event['type']
    result['public'] = event['public']
    
    parsed_timestamp = datetime.fromisoformat(event['created_at'])
    result['create_at'] = parsed_timestamp.timestamp()
    
    result['actor____id'] = event['actor']['id']
    result['actor____login'] = event['actor']['login']
    
    topics = event.get('payload', {}).get('pull_request', {}).get('base', {}).get('repo', {}).get('topics', {})
    
    # now also append all the topics
    processed_topics = []
    for topic in topics:
        processed_topic = {
            'event_id': event['id'],
            'topic_name': topic
        }
        processed_topics.append(processed_topic)
    
    return result, processed_topics

In [ ]:
processed_event = []
processed_topics = []

for event in events_pages:
    processed_event, topics = process_event(event=event)
    process_event.append(processed_event)
    processed_topics.extend(topics)

print(processed_event[:5])
print(processed_topics[:5])

Loaading Data

In [ ]:
import duckdb

# 1. Create a coonecetion to a DuckDB database
conn = duckdb.connect("github_events.db")   # give your database name inside

In [ ]:
# 2. Create the 'github_events' table

conn.execute("""
             CREATE TABLE IF NOT EXISTS github_events( 
             Id TEXT PRIMARY KEY, 
             type TEXT,
             public BOOLEAN, 
             created_at DOUBLE, 
             actor____id BIGINT,
             actor____login TEXT
             """)

In [ ]:
flattened_data = [
    (
        record['id'],
        record['type'],
        record['public'],
        record['create_at'],
        record['actor____id'],
        record['actor____login']
    )
    for record in processed_event[:10]
]

# 3. Insert Data into the github_events table
conn.executemany("""
INSERT INTO github_events (id, type, public, created_at, actor____id, actor____login) 
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT (id) DO NOTHING; """, flattened_data)

In [ ]:
df = conn.execute("SELECT * FROM github_events").df()
df.head()

In [ ]:
conn.close()

Suppose want to add the new coloumn dynamically 

In [ ]:
from datetime import datetime
# datetime.fromisoformat()

def process_event(event):
    result = {}
    
    result['id'] = event['id']
    result['type'] = event['type']
    result['public'] = event['public']
    
    parsed_timestamp = datetime.fromisoformat(event['created_at'])
    result['create_at'] = parsed_timestamp.timestamp()
    
    result['actor____id'] = event['actor']['id']
    result['actor____login'] = event['actor']['login']
    
    result['repo_id'] = event['repo']['id']
    
    topics = event.get('payload', {}).get('pull_request', {}).get('base', {}).get('repo', {}).get('topics', {})
    
    # now also append all the topics
    processed_topics = []
    for topic in topics:
        processed_topic = {
            'event_id': event['id'],
            'topic_name': topic
        }
        processed_topics.append(processed_topic)
    
    return result, processed_topics

In [ ]:
processed_event = []
processed_topics = []

for event in events_pages:
    processed_event, topics = process_event(event=event)
    process_event.append(processed_event)
    processed_topics.extend(topics)

print(processed_event[:5])
print(processed_topics[:5])

In [ ]:
# 1. Create a coonecetion to a DuckDB database
conn = duckdb.connect("github_events.db")   # give your database name inside

In [ ]:
current_columns = {row[1] for row in conn.execute("PRAGMA table_info(github_events)").fetchall()}
print(current_columns)

In [ ]:
# 3. Detect and add new columns dynamically

for record in processed_event[10:]:
    for key in record.key():
        if key not in current_columns:
            col_type = "TEXT"   # Default type()
            if isinstance(record(key), bool):
                col_type = "BOOLEAN"
            elif isinstance(record(key), int):
                col_type = "BIGINT"
            elif isinstance(record(key), float):
                col_type = "DOUBLE"
            print(f"ALTER TABLE github_events ADD COLUMN {key} {col_type};")
            alter_query = f"ALTER TABLE github_events ADD COLUMN {key} {col_type};"
            conn.execute(alter_query)
            print(f"Added new colum: {key} ({col_type})")
            current_columns.add(key)  # update schema tracking

In [ ]:
# 4. Prepare data for insertion (handle missing fields)

columns = sorted(current_columns) # Maintain the consistent order
flattened_data = [
    tuple(record.get(col, None) for col in columns) # Fill missing values with NULL
    for record in processed_event
]

# 5. Construct dynamic SQL for insertion
placeholders = ", ".join(["?" for _ in columns])
columns_str = ", ".join(columns)

insert_query = f"""
INSERT INTO github_events ({columns_str})
VALUES ({placeholders})
ON CONFLICT {id} DO UPDATE SET {", ".join(f"{col}=excluded.{col}" for col in columns if col != 'id')};
"""

In [ ]:
conn.executemany(insert_query, flattened_data)